In [1]:
import duckdb
import pandas as pd

In [2]:
db = duckdb.read_parquet(r"data/input/piebro/changeset_data/year=*/month=*/*.parquet", hive_partitioning=1) 

In [6]:
df_new_contributors = duckdb.sql("""
    WITH user_first_changeset AS (
        SELECT
            user_name,
            year,
            month,
            changeset_id,
            ROW_NUMBER() OVER (
                PARTITION BY user_name 
                ORDER BY year, month, changeset_id
            ) as rn
        FROM db
    )
    SELECT
        user_name,
        changeset_id,
        year,
        month,
        CONCAT(year, '-', LPAD(CAST(month AS VARCHAR), 2, '0')) AS first_edit_month,
    FROM user_first_changeset
    WHERE rn = 1 
    ORDER BY first_edit_month, user_name
""").df()


KeyboardInterrupt



In [ ]:
df_new_contributors.to_csv("../../data/ouput/contribs_01/new_contribs.csv")